In [105]:
import pandas as pd
import numpy as np
import json
import glob

import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim

In [106]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [107]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [108]:
data = load_data("transcripts.json")["Transcripts"]
print (data[0][0:90])
print (data[1][0:90])

Whoa! Okay, yeah. Good. Okay, don’t embarrass yourself, okay. Ohh, the expectations. Crush
I’m just going to start up this show not with a joke, but with a question, that I’d like y


In [109]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:90])

embarrass expectation crushing would argue absolutely crush debilitate way be go move be s


In [124]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hayleyowens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [125]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
print (data_words[1][0:20])

['embarrass', 'expectation', 'crushing', 'would', 'argue', 'absolutely', 'crush', 'debilitate', 'way', 'be', 'go', 'move', 'be', 'so', 'thin', 'will', 'disappear', 'so', 'then', 'will']
['be', 'just', 'go', 'start', 'show', 'joke', 'question', 'answer', 'honestly', 'raise', 'hand', 'here', 'think', 'good', 'person', 'terrifyingly', 'low', 'low', 'put', 'more']


In [126]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['embarrass', 'expectation', 'crushing', 'argue', 'absolutely', 'crush', 'debilitate', 'way', 'move', 'thin', 'disappear', 'right', 'watch', 'comedian', 'stand', 'camera', 'big', 'night', 'really', 'big', 'camera', 'camera', 'everywhere', 'inherently', 'false', 'moment', 'right', 'little', 'bit', 'scary']


In [127]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['embarrass', 'expectation', 'crushing', 'argue', 'absolutely', 'crush', 'debilitate', 'way', 'move', 'thin', 'disappear', 'right', 'watch', 'comedian', 'stand', 'camera', 'big', 'night', 'really', 'big', 'camera', 'camera', 'everywhere', 'inherently', 'false', 'moment', 'right', 'little', 'bit', 'scary', 'film', 'actually', 'film', 'camera', 'inherently', 'misogynist', 'feel', 'attack', 'language', 'shoot', 'shoot', 'get', 'feel', 'scary', 'brave', 'absolutely', 'course', 'force', 'comedy', 'young', 'age', 'bone', 'structure', 'choose', 'stay', 'directness', 'right', 'reach', 'actually', 'change', 'life', 'cry', 'cry', 'sweetie', 'touch', 'sweet', 'sweet', 'sweet', 'look', 'grow', 'home', 'living', 'room', 'friend', 'sweet', 'great', 'moment', 'top', 'unprecedented', 'argue', 'direct', 'physical', 'connection', 'audience', 'aside', 'emotional', 'foster', 'already', 'minute', 'care', 'count', 'breath', 'physical', 'comedy', 'right', 'thing', 'essential', 'comedy', 'primarily', 'process

In [128]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    
    



[(0, 2), (1, 7), (2, 3), (3, 1), (4, 1), (5, 33), (6, 3), (7, 1), (8, 2), (9, 1), (10, 1), (11, 1), (12, 2), (13, 1), (14, 1), (15, 1), (16, 1), (17, 2), (18, 1), (19, 2)]


In [113]:
# id2word = corpora.Dictionary(data_words)

# corpus = []
# for text in data_words:
#     new = id2word.doc2bow(text)
#     corpus.append(new)
    
# print (corpus[0][0:20])

# word = id2word[[0][:1][0]]
# print(word)

In [129]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=30,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [130]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
20    -0.251499 -0.252548       1        1  51.240257
17    -0.037035 -0.397470       2        1  25.337981
18    -0.373490 -0.132598       3        1  23.387914
3      0.020028  0.023589       4        1   0.001408
19     0.022579  0.026880       5        1   0.001258
2      0.022709  0.027135       6        1   0.001258
1      0.023250  0.027656       7        1   0.001256
8      0.023834  0.028167       8        1   0.001255
11     0.024250  0.028639       9        1   0.001255
29     0.024399  0.028748      10        1   0.001254
23     0.024465  0.028963      11        1   0.001253
21     0.024431  0.029032      12        1   0.001253
14     0.024835  0.029327      13        1   0.001252
16     0.024744  0.029267      14        1   0.001252
26     0.024835  0.029295      15        1   0.001251
28     0.024640  0.029014      16        1   0.001251
22     0.024943  0.029388      17        1   0.001251
24     0.024640  0.029062      18        1   0.001251
7      0.024941  0.029516      19        1   0.001251
12     0.025087  0.029671      20        1   0.001250
15     0.024916  0.029339      21        1   0.001249
10     0.025210  0.029740      22        1   0.001249
27     0.025267  0.029856      23        1   0.001248
6      0.025409  0.029994      24        1   0.001248
4      0.025345  0.029948      25        1   0.001247
0      0.025445  0.030067      26        1   0.001247
9      0.025462  0.030061      27        1   0.001246
13     0.025192  0.029771      28        1   0.001246
25     0.025542  0.030205      29        1   0.001204
5      0.025625  0.030287      30        1   0.001204, topic_info=       Term        Freq       Total Category  logprob  loglift
314     get  178.000000  178.000000  Default  30.0000  30.0000
456     mom  112.000000  112.000000  Default  29.0000  29.0000
425    love  134.000000  134.000000  Default  28.0000  28.0000
592   right  136.000000  136.000000  Default  27.0000  27.0000
1049   fuck  100.000000  100.000000  Default  26.0000  26.0000
...     ...         ...         ...      ...      ...      ...
744     try    0.000044   50.496939  Topic30  -7.7233  -2.6351
442    mean    0.000044   58.286247  Topic30  -7.7221  -2.7774
476   never    0.000044   68.173242  Topic30  -7.7208  -2.9328
1235    old    0.000043   39.925536  Topic30  -7.7263  -2.4032
465    much    0.000043   45.724057  Topic30  -7.7258  -2.5384

[1876 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
1         1  1.014873  absolutely
1825      1  1.014881      accept
5         1  0.826701    actually
5         2  0.050614    actually
5         3  0.134972    actually
...     ...       ...         ...
800       2  0.262610        work
800       3  0.350147        work
801       2  1.012823       world
802       3  1.018691       worry
1479      2  0.989782       wrong

[291 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[21, 18, 19, 4, 20, 3, 2, 9, 12, 30, 24, 22, 15, 17, 27, 29, 23, 25, 8, 13, 16, 11, 28, 7, 5, 1, 10, 14, 26, 6])